<a href="https://colab.research.google.com/github/Rifades/Data-Analysis-from-Multiple-Sheet/blob/main/Copy_of_Project_Panaroma_(Marico).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ===========================
# IMPORTS & ENVIRONMENT SETUP
# ===========================

In [ ]:
import pandas as pd
import os
import re
import glob
import time
from pathlib import Path
from datetime import date
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import drive, auth
from google.auth import default

# 1. Mount Drive & Authenticate

In [ ]:
drive.mount('/content/drive')
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

Mounted at /content/drive


# ===========================================
# 1. CORE HELPER FUNCTION (The DRY Principle)
# ===========================================

In [ ]:
def read_and_clean_excel(file_path, required_cols=None):
    try:
        df = pd.read_excel(file_path, skiprows=[0, 1], header=0, engine='openpyxl')

        df.columns = df.columns.str.strip()

        if required_cols:
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                raise KeyError(f"Missing expected columns: {missing_cols}")
            df = df[required_cols]

        return df

    except Exception as e:
        print(f"  [X] Error reading '{Path(file_path).name}': {e}")
        return None

# ============================
# 2. MODULAR REPORT PROCESSORS
# ============================

In [ ]:
def process_stock_report(file_list):
    print(f"\n--- Processing Stock Reports ({len(file_list)} files) ---")
    source_cols = ['PrdDcode', 'PrdName', 'MRP', 'Saleable', 'DisplaySalRate', 'DisplayRate']
    dfs = []
    processed, skipped = 0, 0

    for file_path in file_list:
        path = Path(file_path)
        temp_df = read_and_clean_excel(path, required_cols=source_cols)

        if temp_df is not None:
            # 1. Clean Data
            temp_df = temp_df[temp_df['PrdDcode'].astype(str).str.upper() != 'TOTAL'].copy()
            temp_df = temp_df.dropna(subset=['PrdDcode'])

            # 2. Extract Date & Dist Code from filename
            parts = path.stem.split('_')
            valid_file = False

            if len(parts) == 2:
                file_id = parts[0]
                dist_name = parts[1]
                match = re.search(r'(\d{2})(\d{2})(\d{4})', dist_name)
                if match:
                    month_val, day_val, year_val = int(match.group(1)), int(match.group(2)), int(match.group(3))
                    temp_df['Dist Code'] = file_id
                    temp_df['Date'] = pd.to_datetime(date(year_val, month_val, day_val))
                    valid_file = True
            elif len(parts) == 3:
                temp_df['Dist Code'] = parts[0]
                temp_df['Date'] = pd.to_datetime(parts[1])
                valid_file = True

            # 3. Aggregate
            if valid_file:
                summary_item = temp_df.groupby(['Date', 'Dist Code'], as_index=False).agg({'DisplaySalRate': 'sum'})
                dfs.append(summary_item)
                processed += 1
            else:
                skipped += 1
        else:
            skipped += 1

    print(f"Finished. Processed: {processed}, Skipped: {skipped}")

    if dfs:
        summary_df = pd.concat(dfs, ignore_index=True)
        return summary_df.drop_duplicates(subset=['Date', 'Dist Code'], keep='last').reset_index(drop=True)
    return pd.DataFrame()

In [ ]:
def process_billwise(file_list):
    print(f"\n--- Processing Billwise Items ({len(file_list)} files) ---")
    source_cols = ['Retailer Code', 'Retailer Name', 'Status', 'ChannelName', 'Gross Amt', 'NetAmount', 'Dist Code', 'Delivery Date']
    dfs = []

    for file_path in file_list:
        temp_df = read_and_clean_excel(file_path, required_cols=source_cols)

        if temp_df is not None:
            temp_df = temp_df[temp_df['Status'] == 'Delivered']
            temp_df['Delivery Date'] = pd.to_datetime(temp_df['Delivery Date'], dayfirst=True, errors='coerce').dt.date

            # Aggregate to reduce size
            temp_df = temp_df.groupby(['Dist Code', 'Delivery Date', 'Retailer Code', 'ChannelName'], as_index=False).agg({
                'NetAmount': 'sum', 'Gross Amt': 'sum'
            })
            dfs.append(temp_df)

    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)

        # Split 1: Sales
        sales_df = full_df.groupby(['Dist Code', 'Delivery Date'], as_index=False).agg({'Gross Amt': 'sum'})

        # Split 2: Non-Claim (Wholesale)
        nonclaim_df = full_df[full_df['ChannelName'] == 'Wholesale'].copy()

        return sales_df, nonclaim_df
    return pd.DataFrame(), pd.DataFrame()

In [ ]:
def process_grn_listing(file_list):
    print(f"\n--- Processing GRN Listings ({len(file_list)} files) ---")
    source_cols = ['GRN Date', 'Net Amt.', 'AIT']
    dfs = []

    for file_path in file_list:
        temp_df = read_and_clean_excel(file_path, required_cols=source_cols)

        if temp_df is not None:
            temp_df['GRN Date'] = pd.to_datetime(temp_df['GRN Date'], format='%b %d %Y %I:%M%p', errors='coerce').dt.date
            temp_df.dropna(subset=['GRN Date'], inplace=True)

            parts = Path(file_path).stem.split('_')
            if len(parts) > 1:
                temp_df['Dist Code'] = parts[0]
                dfs.append(temp_df)

    if dfs:
        final_df = pd.concat(dfs, ignore_index=True)
        final_df['Lifting Value'] = final_df['Net Amt.'] - final_df['AIT']
        return final_df.groupby(['Dist Code', 'GRN Date'], as_index=False).agg({
            'Net Amt.': 'sum', 'AIT': 'sum', 'Lifting Value': 'sum'
        })
    return pd.DataFrame()

In [ ]:
def process_claims(file_list):
    print(f"\n--- Processing Claims ({len(file_list)} files) ---")
    source_cols = ['SKU', 'Billed Value']
    dfs = []

    for file_path in file_list:
        temp_df = read_and_clean_excel(file_path, required_cols=source_cols)

        if temp_df is not None:
            # Filter and transpose
            temp_df = temp_df[temp_df['SKU'].isin(['Free Value', 'Total Discount', 'Retailer Credit'])]
            temp_df = temp_df.set_index('SKU').T.reset_index(drop=True)
            temp_df.columns.name = None

            # Extract Dist Code & Date
            parts = Path(file_path).stem.split('_')
            if len(parts) > 1:
                temp_df['Dist Code'] = parts[0]
                # Fix: Call .date() directly on the Timestamp object as it's a scalar, not a Series.
                # Using errors='coerce' will result in NaT if conversion fails,
                # and .date() will raise an AttributeError on NaT.
                # A more robust solution for NaT would involve a check, but for the current error,
                # this directly addresses the .dt accessor issue for valid dates.
                date_val = pd.to_datetime(parts[1], errors="coerce")
                if pd.notna(date_val):
                    temp_df['Date'] = date_val.date()
                else:
                    temp_df['Date'] = pd.NaT # Assign NaT if conversion failed
                dfs.append(temp_df)

    if dfs:
        final_df = pd.concat(dfs, ignore_index=True)
        # Ensure columns exist before summing to prevent KeyError if a file was missing a specific SKU
        for col in ['Free Value', 'Retailer Credit', 'Total Discount']:
            if col not in final_df.columns:
                final_df[col] = 0.0

        final_df['Total Claim'] = final_df['Free Value'] + final_df['Retailer Credit'] + final_df['Total Discount']
        return final_df
    return pd.DataFrame()

# ===================
# 3. EXPORT FUNCTION
# ===================

In [ ]:
def export_multiple_dfs_to_google_sheet(spreadsheet_url, data_dict):
    """Exports data and resizes Google Sheets to stay under cell limits."""
    print("\n--- Exporting to Google Sheets ---")
    try:
        sh = gc.open_by_url(spreadsheet_url)
    except Exception as e:
        print(f"  [X] Could not open spreadsheet: {e}")
        return

    for sheet_name, df in data_dict.items():
        if df.empty:
            print(f"  [-] Skipping {sheet_name}: DataFrame is empty.")
            continue

        try:
            # 1. Get or Create Worksheet
            try:
                worksheet = sh.worksheet(sheet_name)
            except gspread.exceptions.WorksheetNotFound:
                worksheet = sh.add_worksheet(title=sheet_name, rows="1", cols="1")

            # 2. Resize Sheet (CRITICAL for large files)
            rows_needed = len(df) + 1
            cols_needed = len(df.columns)
            print(f"  > Resizing '{sheet_name}' to {rows_needed}x{cols_needed}...")
            worksheet.resize(rows=rows_needed, cols=cols_needed)

            # 3. Clear & Write
            worksheet.clear()
            set_with_dataframe(worksheet, df, include_index=False)
            print(f"  [+] Successfully updated: {sheet_name}")

            # API Rate Limiting Buffer
            time.sleep(2)

        except Exception as e:
            print(f"  [X] Failed to update {sheet_name}: {e}")

    print("\n✅ All exports completed!")

# =======================
# 4. MAIN EXECUTION BLOCK
# =======================

In [ ]:
if __name__ == "__main__":
    # --- A. Setup Variables ---
    base_dir = '/content/drive/My Drive/Marico JBP Files'
    result_date = '22-04-2026'
    calculation_month = 'April 2026'

    sub_folders = {
        'billwise': 'Billwise Itemwise',
        'stock': 'Stock Report',
        'grn': 'GRN Listing',
        'delconstat': 'Delivery Confirmation Statement'
    }

    # --- B. File Discovery ---
    file_lists = {}
    print("--- Searching for Files ---")
    for key, folder_name in sub_folders.items():
        if key in ['stock', 'delconstat']:
            full_path = os.path.join(base_dir, folder_name, calculation_month)
        else:
            full_path = os.path.join(base_dir, folder_name, calculation_month, result_date)

        patterns = [os.path.join(full_path, "**", "*.xlsx"), os.path.join(full_path, "**", "*.xls")]
        files = []
        for p in patterns:
            files.extend(glob.glob(p, recursive=True))

        file_lists[key] = files
        print(f"Found {len(files)} files for '{folder_name}'.")

    # --- C. Data Processing ---
    df_stock = process_stock_report(file_lists['stock'])
    df_sales, df_non_claim = process_billwise(file_lists['billwise'])
    df_grn = process_grn_listing(file_lists['grn'])
    df_claims = process_claims(file_lists['delconstat'])

    # --- D. Consolidation & Export ---
    my_reports = {
        'Stock_Summary': df_stock,
        'Non_Claim': df_non_claim,
        'Inward': df_grn,
        'Claim': df_claims,
        'Sales': df_sales
    }

    target_url = 'https://docs.google.com/spreadsheets/d/1yAt8jd9SuWdenzjGcCeYRzfKosVF9dqoBwKEWysFf94/edit?gid=0#gid=0'
    export_multiple_dfs_to_google_sheet(target_url, my_reports)


--- Searching for Files ---
Found 13 files for 'Billwise Itemwise'.
Found 306 files for 'Stock Report'.
Found 13 files for 'GRN Listing'.
Found 20 files for 'Delivery Confirmation Statement'.

--- Processing Stock Reports (306 files) ---
Finished. Processed: 306, Skipped: 0

--- Processing Billwise Items (13 files) ---

--- Processing GRN Listings (13 files) ---

--- Processing Claims (20 files) ---

--- Exporting to Google Sheets ---
  > Resizing 'Stock_Summary' to 274x3...
  [+] Successfully updated: Stock_Summary
  > Resizing 'Non_Claim' to 1079x6...
  [+] Successfully updated: Non_Claim
  > Resizing 'Inward' to 36x5...
  [+] Successfully updated: Inward
  > Resizing 'Claim' to 21x6...
  [+] Successfully updated: Claim
  > Resizing 'Sales' to 233x3...
  [+] Successfully updated: Sales

✅ All exports completed!
